# HUMAN-AI INTERACTION GENOME PROJECT

SETUP & LIBRARY IMPORTS

In [42]:
!pip install bertopic umap-learn # bertopic requires umap-learn

In [43]:
# Core Data Processing
import pandas as pd
import numpy as np
import json
import re
import hashlib
from datetime import datetime
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# NLP & Text Processing
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

# Download required NLTK data
nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

# Advanced NLP with Transformers
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# Topic Modeling
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

# Machine Learning & Clustering
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score
from minisom import MiniSom

# Dimensionality Reduction
import umap.umap_ as umap

# Statistical Modeling
import statsmodels.api as sm
from statsmodels.formula.api import ols
from scipy import stats
from scipy.stats import chi2_contingency

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✅ All libraries imported successfully!")
print("=" * 80)

✅ All libraries imported successfully!


DATA LOADING & INITIAL EXPLORATION

In [44]:
print("\n📊 SECTION 2: DATA LOADING & INITIAL EXPLORATION")
print("=" * 80)

# Dataset Information
print("""
RECOMMENDED DATASETS FOR THIS PROJECT:
======================================

1. PRIMARY DATASET - WildChat (Allen AI)
   - Source: https://huggingface.co/datasets/allenai/WildChat-1M
   - Size: 1 million conversations, 2.5M+ interaction turns
   - Features: Timestamped conversations, demographics, user prompts, AI responses
   - Languages: Multi-lingual (66+ languages)
   - Download: Use Hugging Face datasets library

2. SUPPLEMENTARY DATASETS:

   a) UltraFeedback (OpenBMB)
      - Source: https://huggingface.co/datasets/openbmb/UltraFeedback
      - Features: User feedback, ratings, helpfulness scores

   b) Anthropic HH-RLHF
      - Source: https://huggingface.co/datasets/Anthropic/hh-rlhf
      - Features: Human feedback on AI responses

   c) ShareGPT Conversations
      - Source: https://huggingface.co/datasets/anon8231489123/ShareGPT_Vicuna_unfiltered
      - Features: Real user conversations with ChatGPT

   d) LMSYS Chat1M
      - Source: https://huggingface.co/datasets/lmsys/lmsys-chat-1m
      - Features: Chat data from multiple AI models

SIMULATED DATA GENERATION (For demonstration):
==============================================
Since actual datasets require download, we'll generate realistic synthetic data
that matches the structure and characteristics of the WildChat dataset.
""")

def generate_synthetic_wildchat_data(n_conversations=5000, n_turns_range=(1, 15)):
    """
    Generate synthetic conversation data mimicking WildChat structure

    Parameters:
    -----------
    n_conversations : int
        Number of unique conversations to generate
    n_turns_range : tuple
        Min and max number of turns per conversation

    Returns:
    --------
    pd.DataFrame : Synthetic conversation dataset
    """
    np.random.seed(42)

    # Sample conversation templates and patterns
    user_intents = [
        'information_seeking', 'creative_writing', 'coding_help',
        'advice_seeking', 'brainstorming', 'explanation_request',
        'problem_solving', 'casual_chat', 'learning', 'task_completion'
    ]

    emotions = ['neutral', 'curious', 'frustrated', 'satisfied', 'confused',
                'excited', 'anxious', 'grateful', 'disappointed']

    countries = ['US', 'UK', 'IN', 'CA', 'AU', 'DE', 'FR', 'BR', 'JP', 'CN']

    conversation_data = []

    for conv_id in range(n_conversations):
        n_turns = np.random.randint(n_turns_range[0], n_turns_range[1])
        user_id = f"user_{hashlib.md5(str(conv_id).encode()).hexdigest()[:8]}"
        country = np.random.choice(countries)

        # Simulate user engagement pattern
        base_engagement = np.random.uniform(0.3, 0.9)
        base_satisfaction = np.random.uniform(0.2, 0.8)

        for turn_idx in range(n_turns):
            timestamp = pd.Timestamp('2024-01-01') + pd.Timedelta(days=np.random.randint(0, 365))

            # User characteristics that evolve over conversation
            confusion_level = max(0, min(1, np.random.beta(2, 5) + (turn_idx * 0.02)))
            trust_level = max(0, min(1, base_satisfaction + np.random.normal(0, 0.1)))
            engagement_time = np.random.exponential(scale=120) + 30  # seconds

            # Simulate sentiment drift over conversation
            sentiment_compound = np.random.normal(
                loc=base_satisfaction * 2 - 1,  # Map 0-1 to -1 to 1
                scale=0.3
            )
            sentiment_compound = max(-1, min(1, sentiment_compound))

            conversation_data.append({
                'conversation_id': conv_id,
                'turn_number': turn_idx,
                'user_id': user_id,
                'timestamp': timestamp,
                'country': country,
                'user_intent': np.random.choice(user_intents),
                'user_emotion': np.random.choice(emotions, p=[0.25, 0.15, 0.1, 0.15, 0.1, 0.08, 0.07, 0.05, 0.05]),
                'user_message_length': int(np.random.lognormal(mean=4, sigma=1)),
                'ai_response_length': int(np.random.lognormal(mean=5, sigma=1)),
                'engagement_time_sec': engagement_time,
                'confusion_score': confusion_level,
                'trust_score': trust_level,
                'satisfaction_score': base_satisfaction + np.random.normal(0, 0.15) * (1 - turn_idx/n_turns),
                'sentiment_compound': sentiment_compound,
                'sentiment_positive': max(0, sentiment_compound),
                'sentiment_negative': max(0, -sentiment_compound),
                'sentiment_neutral': 1 - abs(sentiment_compound),
                'follow_up': 1 if turn_idx < n_turns - 1 else 0,
                'user_feedback_provided': np.random.choice([0, 1], p=[0.85, 0.15])
            })

    df = pd.DataFrame(conversation_data)

    # Add derived features
    df['hour_of_day'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

    return df

# Generate synthetic dataset
print("\n🔄 Generating synthetic conversation data...")
df_conversations = generate_synthetic_wildchat_data(n_conversations=5000, n_turns_range=(1, 15))

print(f"✅ Generated {len(df_conversations)} conversation turns")
print(f"   - Unique conversations: {df_conversations['conversation_id'].nunique()}")
print(f"   - Unique users: {df_conversations['user_id'].nunique()}")
print(f"   - Date range: {df_conversations['timestamp'].min()} to {df_conversations['timestamp'].max()}")

print("\n📋 Dataset Preview:")
print(df_conversations.head(10))

print("\n📊 Dataset Summary Statistics:")
print(df_conversations.describe())

print("=" * 80)


📊 SECTION 2: DATA LOADING & INITIAL EXPLORATION

RECOMMENDED DATASETS FOR THIS PROJECT:

1. PRIMARY DATASET - WildChat (Allen AI)
   - Source: https://huggingface.co/datasets/allenai/WildChat-1M
   - Size: 1 million conversations, 2.5M+ interaction turns
   - Features: Timestamped conversations, demographics, user prompts, AI responses
   - Languages: Multi-lingual (66+ languages)
   - Download: Use Hugging Face datasets library

2. SUPPLEMENTARY DATASETS:

   a) UltraFeedback (OpenBMB)
      - Source: https://huggingface.co/datasets/openbmb/UltraFeedback
      - Features: User feedback, ratings, helpfulness scores

   b) Anthropic HH-RLHF
      - Source: https://huggingface.co/datasets/Anthropic/hh-rlhf
      - Features: Human feedback on AI responses

   c) ShareGPT Conversations
      - Source: https://huggingface.co/datasets/anon8231489123/ShareGPT_Vicuna_unfiltered
      - Features: Real user conversations with ChatGPT

   d) LMSYS Chat1M
      - Source: https://huggingface.co/da

DATA PREPROCESSING & ANONYMIZATION

In [45]:
print("\n🔒 SECTION 3: DATA PREPROCESSING & ANONYMIZATION")
print("=" * 80)

def anonymize_user_data(df):
    """
    Ensure all user identifiers are properly anonymized
    """
    df_anon = df.copy()

    # Hash user IDs if not already hashed
    df_anon['user_id_hashed'] = df_anon['user_id'].apply(
        lambda x: hashlib.sha256(str(x).encode()).hexdigest()[:16]
    )

    # Remove original user_id
    df_anon = df_anon.drop(columns=['user_id'])

    print("✅ User data anonymized successfully")
    return df_anon

def clean_and_normalize_data(df):
    """
    Clean and normalize the dataset
    """
    df_clean = df.copy()

    # Handle missing values
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
    df_clean[numeric_cols] = df_clean[numeric_cols].fillna(df_clean[numeric_cols].median())

    # Cap outliers for engagement time (> 99th percentile)
    engagement_cap = df_clean['engagement_time_sec'].quantile(0.99)
    df_clean['engagement_time_sec'] = df_clean['engagement_time_sec'].clip(upper=engagement_cap)

    # Ensure scores are in valid range [0, 1]
    score_cols = ['confusion_score', 'trust_score', 'satisfaction_score']
    for col in score_cols:
        df_clean[col] = df_clean[col].clip(0, 1)

    print("✅ Data cleaned and normalized")
    return df_clean

# Apply preprocessing
df_processed = anonymize_user_data(df_conversations)
df_processed = clean_and_normalize_data(df_processed)

print(f"\n✅ Preprocessing complete. Final dataset shape: {df_processed.shape}")
print("=" * 80)


🔒 SECTION 3: DATA PREPROCESSING & ANONYMIZATION
✅ User data anonymized successfully
✅ Data cleaned and normalized

✅ Preprocessing complete. Final dataset shape: (37656, 22)


ADVANCED FEATURE ENGINEERING

In [46]:
print("\n🔧 SECTION 4: ADVANCED FEATURE ENGINEERING")
print("=" * 80)

def extract_linguistic_features(df):
    """
    Extract NLP-based linguistic features
    """
    print("\n📝 Extracting linguistic features...")

    df_features = df.copy()

    # Sentiment intensity (already in synthetic data, but would use VADER in real implementation)
    # For real data: sia = SentimentIntensityAnalyzer()

    # Message complexity features
    df_features['avg_message_words'] = df_features['user_message_length'] / (df_features['turn_number'] + 1)
    df_features['response_length_ratio'] = df_features['ai_response_length'] / (df_features['user_message_length'] + 1)

    # Conversation dynamics
    df_features['turn_position_normalized'] = df_features.groupby('conversation_id')['turn_number'].transform(
        lambda x: x / x.max() if x.max() > 0 else 0
    )

    print("✅ Linguistic features extracted")
    return df_features

def extract_behavioral_features(df):
    """
    Extract behavioral engagement features
    """
    print("\n👤 Extracting behavioral features...")

    df_features = df.copy()

    # Engagement metrics
    df_features['engagement_intensity'] = np.log1p(df_features['engagement_time_sec'])

    # User persistence
    conversation_stats = df_features.groupby('conversation_id').agg({
        'turn_number': 'max',
        'engagement_time_sec': 'sum',
        'follow_up': 'sum'
    }).rename(columns={
        'turn_number': 'total_turns',
        'engagement_time_sec': 'total_engagement',
        'follow_up': 'num_follow_ups'
    })

    df_features = df_features.merge(
        conversation_stats,
        left_on='conversation_id',
        right_index=True,
        how='left'
    )

    # Interaction rhythm
    df_features['avg_time_per_turn'] = df_features['total_engagement'] / (df_features['total_turns'] + 1)

    print("✅ Behavioral features extracted")
    return df_features

def extract_emotional_features(df):
    """
    Extract emotion and affect-based features
    """
    print("\n💭 Extracting emotional features...")

    df_features = df.copy()

    # Emotion category encoding (one-hot)
    emotion_dummies = pd.get_dummies(df_features['user_emotion'], prefix='emotion')
    df_features = pd.concat([df_features, emotion_dummies], axis=1)

    # Emotional trajectory
    df_features['sentiment_trajectory'] = df_features.groupby('conversation_id')['sentiment_compound'].transform(
        lambda x: x.diff().fillna(0)
    )

    # Emotional volatility within conversation
    df_features['sentiment_volatility'] = df_features.groupby('conversation_id')['sentiment_compound'].transform('std').fillna(0)

    print("✅ Emotional features extracted")
    return df_features

def extract_contextual_features(df):
    """
    Extract temporal and contextual features
    """
    print("\n🌐 Extracting contextual features...")

    df_features = df.copy()

    # Temporal patterns
    df_features['is_business_hours'] = df_features['hour_of_day'].between(9, 17).astype(int)
    df_features['is_night'] = df_features['hour_of_day'].between(22, 24) | df_features['hour_of_day'].between(0, 6)
    df_features['is_night'] = df_features['is_night'].astype(int)

    # Geographic (country) encoding
    country_dummies = pd.get_dummies(df_features['country'], prefix='country')
    df_features = pd.concat([df_features, country_dummies], axis=1)

    # Intent category encoding
    intent_dummies = pd.get_dummies(df_features['user_intent'], prefix='intent')
    df_features = pd.concat([df_features, intent_dummies], axis=1)

    print("✅ Contextual features extracted")
    return df_features

# Apply all feature engineering
df_engineered = extract_linguistic_features(df_processed)
df_engineered = extract_behavioral_features(df_engineered)
df_engineered = extract_emotional_features(df_engineered)
df_engineered = extract_contextual_features(df_engineered)

print(f"\n✅ Feature engineering complete. Total features: {df_engineered.shape[1]}")
print("=" * 80)


🔧 SECTION 4: ADVANCED FEATURE ENGINEERING

📝 Extracting linguistic features...
✅ Linguistic features extracted

👤 Extracting behavioral features...
✅ Behavioral features extracted

💭 Extracting emotional features...
✅ Emotional features extracted

🌐 Extracting contextual features...
✅ Contextual features extracted

✅ Feature engineering complete. Total features: 63


UNIFIED FEATURE REPRESENTATION

In [47]:
print("\n🎯 SECTION 5: UNIFIED FEATURE REPRESENTATION")
print("=" * 80)

def create_unified_feature_matrix(df):
    """
    Create unified feature matrix for clustering and archetype discovery
    """
    print("\n🔗 Creating unified feature matrix...")

    # Select key features for behavioral genome
    behavioral_features = [
        'engagement_time_sec', 'engagement_intensity', 'avg_time_per_turn',
        'total_turns', 'num_follow_ups'
    ]

    emotional_features = [
        'sentiment_compound', 'sentiment_positive', 'sentiment_negative',
        'sentiment_trajectory', 'sentiment_volatility'
    ]

    feedback_features = [
        'confusion_score', 'trust_score', 'satisfaction_score'
    ]

    linguistic_features = [
        'user_message_length', 'ai_response_length',
        'avg_message_words', 'response_length_ratio',
        'turn_position_normalized'
    ]

    contextual_features = [
        'hour_of_day', 'is_weekend', 'is_business_hours', 'is_night'
    ]

    # Collect all emotion dummies
    emotion_cols = [col for col in df.columns if col.startswith('emotion_')]

    # Collect all intent dummies
    intent_cols = [col for col in df.columns if col.startswith('intent_')]

    # Combine all feature groups
    all_features = (behavioral_features + emotional_features + feedback_features +
                   linguistic_features + contextual_features + emotion_cols + intent_cols)

    # Ensure all features exist
    available_features = [f for f in all_features if f in df.columns]

    X = df[available_features].copy()

    # Handle any remaining NaN
    X = X.fillna(X.median())

    print(f"✅ Unified matrix created with {X.shape[1]} features")
    return X, available_features

X_features, feature_names = create_unified_feature_matrix(df_engineered)

# Normalize features for clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_names, index=X_features.index)

print(f"✅ Features normalized. Matrix shape: {X_scaled_df.shape}")
print("=" * 80)


🎯 SECTION 5: UNIFIED FEATURE REPRESENTATION

🔗 Creating unified feature matrix...
✅ Unified matrix created with 41 features
✅ Features normalized. Matrix shape: (37656, 41)


PATTERN DISCOVERY - SELF-ORGANIZING MAPS (SOM)

In [48]:
print("\n🗺️ SECTION 6: PATTERN DISCOVERY - SELF-ORGANIZING MAPS")
print("=" * 80)

def train_self_organizing_map(X, map_size=(10, 10), sigma=1.0, learning_rate=0.5, iterations=10000):
    """
    Train Self-Organizing Map to discover behavioral archetypes

    Parameters:
    -----------
    X : array-like
        Feature matrix
    map_size : tuple
        Dimensions of SOM grid
    sigma : float
        Spread of neighborhood function
    learning_rate : float
        Initial learning rate
    iterations : int
        Number of training iterations

    Returns:
    --------
    MiniSom object and cluster assignments
    """
    print(f"\n🧠 Training SOM with grid size {map_size}...")

    n_features = X.shape[1]

    # Initialize SOM
    som = MiniSom(
        x=map_size[0],
        y=map_size[1],
        input_len=n_features,
        sigma=sigma,
        learning_rate=learning_rate,
        random_seed=42
    )

    # Initialize weights
    som.random_weights_init(X)

    # Train
    som.train_batch(X, iterations, verbose=False)

    print(f"✅ SOM training complete after {iterations} iterations")

    # Map each sample to winning neuron
    winner_coordinates = np.array([som.winner(x) for x in X])

    # Convert 2D coordinates to cluster IDs
    cluster_ids = winner_coordinates[:, 0] * map_size[1] + winner_coordinates[:, 1]

    print(f"✅ Identified {len(np.unique(cluster_ids))} unique behavioral patterns")

    return som, cluster_ids, winner_coordinates

# Train SOM
som_model, som_clusters, som_coords = train_self_organizing_map(
    X_scaled,
    map_size=(8, 8),
    sigma=1.5,
    learning_rate=0.5,
    iterations=15000
)

# Add SOM clusters to dataframe
df_engineered['som_cluster'] = som_clusters

print(f"\n📊 SOM Cluster Distribution:")
print(df_engineered['som_cluster'].value_counts().head(15))
print("=" * 80)


🗺️ SECTION 6: PATTERN DISCOVERY - SELF-ORGANIZING MAPS

🧠 Training SOM with grid size (8, 8)...
✅ SOM training complete after 15000 iterations
✅ Identified 64 unique behavioral patterns

📊 SOM Cluster Distribution:
som_cluster
18    1382
49    1064
60    1017
20    1010
57    1005
51     930
31     921
36     909
30     857
38     808
17     798
33     791
54     769
63     761
11     759
Name: count, dtype: int64


DIMENSIONALITY REDUCTION & CLUSTERING VALIDATION

In [49]:
print("\n🔍 SECTION 7: DIMENSIONALITY REDUCTION & CLUSTERING")
print("=" * 80)

def apply_umap_reduction(X, n_components=2, n_neighbors=15, min_dist=0.1):
    """
    Apply UMAP for dimensionality reduction and visualization
    """
    print(f"\n📉 Applying UMAP reduction to {n_components} dimensions...")

    reducer = umap.UMAP(
        n_components=n_components,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric='euclidean',
        random_state=42
    )

    X_umap = reducer.fit_transform(X)

    print(f"✅ UMAP reduction complete")
    return X_umap, reducer

# Apply UMAP
X_umap, umap_reducer = apply_umap_reduction(X_scaled, n_components=2)

# Add UMAP coordinates to dataframe
df_engineered['umap_1'] = X_umap[:, 0]
df_engineered['umap_2'] = X_umap[:, 1]

# Additional clustering with KMeans for comparison
print("\n🎯 Applying KMeans clustering...")
n_clusters_optimal = 8
kmeans = KMeans(n_clusters=n_clusters_optimal, random_state=42, n_init=10)
df_engineered['kmeans_cluster'] = kmeans.fit_predict(X_scaled)

# Evaluate clustering quality
silhouette_som = silhouette_score(X_scaled, som_clusters)
silhouette_kmeans = silhouette_score(X_scaled, df_engineered['kmeans_cluster'])

print(f"\n📊 Clustering Quality Metrics:")
print(f"   - SOM Silhouette Score: {silhouette_som:.4f}")
print(f"   - KMeans Silhouette Score: {silhouette_kmeans:.4f}")
print("=" * 80)


🔍 SECTION 7: DIMENSIONALITY REDUCTION & CLUSTERING

📉 Applying UMAP reduction to 2 dimensions...
✅ UMAP reduction complete

🎯 Applying KMeans clustering...

📊 Clustering Quality Metrics:
   - SOM Silhouette Score: 0.0522
   - KMeans Silhouette Score: 0.0850


BEHAVIORAL ARCHETYPE CHARACTERIZATION

In [50]:
print("\n🎭 SECTION 8: BEHAVIORAL ARCHETYPE CHARACTERIZATION")
print("=" * 80)

def characterize_archetypes(df, cluster_col='som_cluster', top_n=10):
    """
    Characterize and name behavioral archetypes based on cluster properties
    """
    print(f"\n🔬 Characterizing archetypes from {cluster_col}...")

    # Key behavioral metrics for characterization
    behavioral_metrics = [
        'engagement_time_sec', 'trust_score', 'satisfaction_score',
        'confusion_score', 'sentiment_compound', 'total_turns',
        'user_message_length'
    ]

    # Calculate cluster profiles
    cluster_profiles = df.groupby(cluster_col)[behavioral_metrics].mean()
    cluster_sizes = df[cluster_col].value_counts()

    # Get top N largest clusters
    top_clusters = cluster_sizes.head(top_n).index

    print(f"\n📋 Top {top_n} Behavioral Archetypes:")
    print("=" * 60)

    archetype_names = {}

    for cluster_id in top_clusters:
        profile = cluster_profiles.loc[cluster_id]
        size = cluster_sizes.loc[cluster_id]

        # Characterize based on profile
        engagement_level = "High" if profile['engagement_time_sec'] > df['engagement_time_sec'].median() else "Low"
        satisfaction_level = "High" if profile['satisfaction_score'] > 0.6 else "Moderate" if profile['satisfaction_score'] > 0.4 else "Low"
        confusion_level = "High" if profile['confusion_score'] > 0.5 else "Low"
        sentiment_tone = "Positive" if profile['sentiment_compound'] > 0.2 else "Negative" if profile['sentiment_compound'] < -0.2 else "Neutral"

        # Generate archetype name
        if engagement_level == "High" and satisfaction_level == "High":
            archetype = "🎯 Engaged Enthusiast"
        elif confusion_level == "High":
            archetype = "❓ Confused Explorer"
        elif engagement_level == "Low" and satisfaction_level == "Low":
            archetype = "😐 Disengaged User"
        elif profile['total_turns'] > df['total_turns'].quantile(0.75):
            archetype = "💬 Persistent Questioner"
        elif sentiment_tone == "Positive" and satisfaction_level == "High":
            archetype = "😊 Satisfied User"
        elif sentiment_tone == "Negative":
            archetype = "😞 Frustrated User"
        elif profile['user_message_length'] > df['user_message_length'].quantile(0.75):
            archetype = "📝 Detailed Communicator"
        else:
            archetype = f"👤 Mixed Pattern {cluster_id}"

        archetype_names[cluster_id] = archetype

        print(f"\nCluster {cluster_id}: {archetype}")
        print(f"   Size: {size} interactions ({size/len(df)*100:.1f}%)")
        print(f"   Engagement: {engagement_level} ({profile['engagement_time_sec']:.1f}s)")
        print(f"   Satisfaction: {satisfaction_level} ({profile['satisfaction_score']:.3f})")
        print(f"   Confusion: {confusion_level} ({profile['confusion_score']:.3f})")
        print(f"   Sentiment: {sentiment_tone} ({profile['sentiment_compound']:.3f})")
        print(f"   Avg Turns: {profile['total_turns']:.1f}")

    return archetype_names, cluster_profiles

archetype_names, archetype_profiles = characterize_archetypes(df_engineered, cluster_col='som_cluster', top_n=10)

# Add archetype names to dataframe
df_engineered['archetype_name'] = df_engineered['som_cluster'].map(archetype_names)
df_engineered['archetype_name'] = df_engineered['archetype_name'].fillna("Other Pattern")

print("=" * 80)


🎭 SECTION 8: BEHAVIORAL ARCHETYPE CHARACTERIZATION

🔬 Characterizing archetypes from som_cluster...

📋 Top 10 Behavioral Archetypes:

Cluster 18: 👤 Mixed Pattern 18
   Size: 1382 interactions (3.7%)
   Engagement: High (143.6s)
   Satisfaction: Moderate (0.548)
   Confusion: Low (0.369)
   Sentiment: Neutral (0.135)
   Avg Turns: 8.4

Cluster 49: 😞 Frustrated User
   Size: 1064 interactions (2.8%)
   Engagement: High (157.4s)
   Satisfaction: Low (0.347)
   Confusion: Low (0.387)
   Sentiment: Negative (-0.473)
   Avg Turns: 9.4

Cluster 60: 👤 Mixed Pattern 60
   Size: 1017 interactions (2.7%)
   Engagement: High (166.6s)
   Satisfaction: Moderate (0.509)
   Confusion: Low (0.429)
   Sentiment: Neutral (0.023)
   Avg Turns: 10.0

Cluster 20: 🎯 Engaged Enthusiast
   Size: 1010 interactions (2.7%)
   Engagement: High (191.7s)
   Satisfaction: High (0.650)
   Confusion: Low (0.374)
   Sentiment: Positive (0.373)
   Avg Turns: 9.4

Cluster 57: 👤 Mixed Pattern 57
   Size: 1005 interactions

TOPIC MODELING WITH BERTOPIC

In [51]:

print("\n📚 SECTION 9: TOPIC MODELING (SIMULATED)")
print("=" * 80)

print("""
\n💡 Topic Modeling with BERTopic:
==================================

In a full implementation with real conversation text, we would:

1. Extract conversation text from user_message and ai_response fields
2. Preprocess and clean the text
3. Use BERTopic to discover latent topics within each archetype:

   from bertopic import BERTopic
   from sentence_transformers import SentenceTransformer

   # Create embeddings
   embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

   # Fit BERTopic per archetype
   for archetype in archetypes:
       texts = df[df['archetype']==archetype]['conversation_text']
       topic_model = BERTopic(embedding_model=embedding_model)
       topics, probs = topic_model.fit_transform(texts)

4. Analyze topic distributions across archetypes
5. Extract representative keywords and phrases

For this demonstration, we'll simulate topic assignments:
""")

# Simulate topic assignments
np.random.seed(42)
n_topics = 15
topic_names = [
    'Technical Support', 'Creative Writing', 'Career Advice',
    'Coding Help', 'General Knowledge', 'Math & Science',
    'Product Recommendations', 'Health & Wellness', 'Travel Planning',
    'Business Strategy', 'Learning & Education', 'Entertainment',
    'Personal Development', 'News & Current Events', 'Other'
]

df_engineered['dominant_topic'] = np.random.choice(range(n_topics), size=len(df_engineered))
df_engineered['topic_name'] = df_engineered['dominant_topic'].map(lambda x: topic_names[x] if x < len(topic_names) else 'Other')

print("\n✅ Topic modeling (simulated) complete")
print(f"\n📊 Topic Distribution:")
print(df_engineered['topic_name'].value_counts().head(10))
print("=" * 80)


📚 SECTION 9: TOPIC MODELING (SIMULATED)


💡 Topic Modeling with BERTopic:

In a full implementation with real conversation text, we would:

1. Extract conversation text from user_message and ai_response fields
2. Preprocess and clean the text
3. Use BERTopic to discover latent topics within each archetype:

   from bertopic import BERTopic
   from sentence_transformers import SentenceTransformer

   # Create embeddings
   embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

   # Fit BERTopic per archetype
   for archetype in archetypes:
       texts = df[df['archetype']==archetype]['conversation_text']
       topic_model = BERTopic(embedding_model=embedding_model)
       topics, probs = topic_model.fit_transform(texts)

4. Analyze topic distributions across archetypes
5. Extract representative keywords and phrases

For this demonstration, we'll simulate topic assignments:


✅ Topic modeling (simulated) complete

📊 Topic Distribution:
topic_name
Personal Development       2596
Te

TEMPORAL ANALYSIS & MARKOV CHAIN TRANSITIONS

In [52]:
print("\n⏱️ SECTION 10: TEMPORAL ANALYSIS & BEHAVIORAL TRANSITIONS")
print("=" * 80)

def analyze_archetype_stability(df):
    """
    Analyze how users transition between archetypes over time
    """
    print("\n🔄 Analyzing archetype transitions...")

    # For each user, track archetype changes across conversations
    user_archetype_sequences = []

    for user in df['user_id_hashed'].unique():
        user_data = df[df['user_id_hashed'] == user].sort_values('timestamp')
        if len(user_data) > 1:
            archetypes = user_data['som_cluster'].values
            user_archetype_sequences.append(archetypes)

    print(f"✅ Tracked {len(user_archetype_sequences)} users with multiple interactions")

    return user_archetype_sequences

def build_transition_matrix(sequences, n_states):
    """
    Build Markov chain transition matrix from archetype sequences
    """
    print(f"\n🎲 Building transition matrix for {n_states} states...")

    transition_counts = np.zeros((n_states, n_states))

    for sequence in sequences:
        for i in range(len(sequence) - 1):
            from_state = sequence[i]
            to_state = sequence[i + 1]
            if from_state < n_states and to_state < n_states:
                transition_counts[from_state, to_state] += 1

    # Normalize to get probabilities
    row_sums = transition_counts.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1  # Avoid division by zero
    transition_probs = transition_counts / row_sums

    print("✅ Transition matrix constructed")
    return transition_probs, transition_counts

# Analyze transitions
archetype_sequences = analyze_archetype_stability(df_engineered)

# Build transition matrix for top archetypes
n_top_archetypes = 10
transition_matrix, transition_counts = build_transition_matrix(
    archetype_sequences,
    n_states=n_top_archetypes
)

print(f"\n📊 Transition Matrix Shape: {transition_matrix.shape}")
print(f"\n🔝 Most Common Transitions:")

# Find top transitions
top_transitions = []
for i in range(n_top_archetypes):
    for j in range(n_top_archetypes):
        if transition_counts[i, j] > 0:
            top_transitions.append((i, j, transition_counts[i, j], transition_matrix[i, j]))

top_transitions = sorted(top_transitions, key=lambda x: x[2], reverse=True)[:10]

for from_arch, to_arch, count, prob in top_transitions:
    from_name = archetype_names.get(from_arch, f"Archetype {from_arch}")
    to_name = archetype_names.get(to_arch, f"Archetype {to_arch}")
    print(f"   {from_name} → {to_name}: {int(count)} transitions (p={prob:.3f})")

print("=" * 80)


⏱️ SECTION 10: TEMPORAL ANALYSIS & BEHAVIORAL TRANSITIONS

🔄 Analyzing archetype transitions...
✅ Tracked 4629 users with multiple interactions

🎲 Building transition matrix for 10 states...
✅ Transition matrix constructed

📊 Transition Matrix Shape: (10, 10)

🔝 Most Common Transitions:
   Archetype 3 → Archetype 3: 27 transitions (p=0.255)
   Archetype 2 → Archetype 8: 23 transitions (p=0.187)
   Archetype 2 → Archetype 2: 22 transitions (p=0.179)
   Archetype 0 → Archetype 3: 20 transitions (p=0.215)
   Archetype 9 → Archetype 9: 20 transitions (p=0.541)
   Archetype 2 → Archetype 5: 18 transitions (p=0.146)
   Archetype 0 → Archetype 0: 17 transitions (p=0.183)
   Archetype 8 → Archetype 2: 17 transitions (p=0.157)
   Archetype 3 → Archetype 2: 16 transitions (p=0.151)
   Archetype 8 → Archetype 0: 16 transitions (p=0.148)


STATISTICAL MODELING & REGRESSION ANALYSIS

In [53]:
print("\n📈 SECTION 11: STATISTICAL MODELING & REGRESSION ANALYSIS")
print("=" * 80)

def regression_satisfaction_model(df):
    """
    Build regression model to predict satisfaction from behavioral factors
    """
    print("\n📊 Building satisfaction prediction model...")

    # Select predictors
    predictors = [
        'engagement_time_sec', 'confusion_score', 'sentiment_compound',
        'total_turns', 'trust_score', 'user_message_length'
    ]

    # Prepare data
    X_reg = df[predictors].copy()
    y_reg = df['satisfaction_score'].copy()

    # Remove any NaN
    valid_idx = ~(X_reg.isna().any(axis=1) | y_reg.isna())
    X_reg = X_reg[valid_idx]
    y_reg = y_reg[valid_idx]

    # Add constant for intercept
    X_reg_const = sm.add_constant(X_reg)

    # Fit OLS model
    model = sm.OLS(y_reg, X_reg_const).fit()

    print("\n✅ Regression model fitted")
    print("\n📋 Model Summary:")
    print(model.summary())

    return model

# Build regression model
satisfaction_model = regression_satisfaction_model(df_engineered)

# Correlation analysis
print("\n🔗 Key Correlations with Satisfaction:")
correlation_vars = ['engagement_time_sec', 'confusion_score', 'sentiment_compound',
                    'trust_score', 'total_turns']
for var in correlation_vars:
    corr = df_engineered[[var, 'satisfaction_score']].corr().iloc[0, 1]
    print(f"   {var}: r = {corr:.3f}")

print("=" * 80)


📈 SECTION 11: STATISTICAL MODELING & REGRESSION ANALYSIS

📊 Building satisfaction prediction model...

✅ Regression model fitted

📋 Model Summary:
                            OLS Regression Results                            
Dep. Variable:     satisfaction_score   R-squared:                       0.619
Model:                            OLS   Adj. R-squared:                  0.619
Method:                 Least Squares   F-statistic:                 1.020e+04
Date:                Thu, 06 Nov 2025   Prob (F-statistic):               0.00
Time:                        00:34:49   Log-Likelihood:                 26341.
No. Observations:               37656   AIC:                        -5.267e+04
Df Residuals:                   37649   BIC:                        -5.261e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t

POWER BI DASHBOARD DATA PREPARATION

In [54]:
def assign_behavioral_archetype_names(archetype_stats_df):
    """Assign meaningful names based on behavioral profiles"""

    def name_archetype(row):
        engagement = row['engagement_time_sec_mean']
        confusion = row['confusion_score_mean']
        trust = row['trust_score_mean']
        satisfaction = row['satisfaction_score_mean']
        sentiment = row['sentiment_compound_mean']
        turns = row['total_turns_mean']

        # High engagement + high trust + positive sentiment
        if engagement > 160 and trust > 0.65 and sentiment > 0.3:
            return '🎯 Engaged Enthusiast'

        # Low trust + negative sentiment
        if trust < 0.40 and sentiment < -0.3:
            return '😞 Frustrated User'

        # High confusion + many turns
        if confusion > 0.43 and turns > 10:
            return '❓ Confused Explorer'

        # High engagement + high confusion
        if engagement > 160 and confusion > 0.42:
            return '🤔 Persistent Seeker'

        # Low engagement + low trust
        if engagement < 105 and trust < 0.46:
            return '🚶 Passive Observer'

        # Low confusion + high trust
        if confusion < 0.37 and trust > 0.52:
            return '✅ Efficient Collaborator'

        # Many turns + positive sentiment
        if turns > 9 and sentiment > 0.1:
            return '📚 Curious Learner'

        # High satisfaction
        if satisfaction > 0.65:
            return '👍 Satisfied User'

        # Positive sentiment + exploration
        if sentiment > 0.15 and turns > 8:
            return '😊 Optimistic Explorer'

        # Low trust + critical
        if trust < 0.45 and sentiment < 0:
            return '🧐 Skeptical Critic'

        # Default: balanced behavior
        return '⚖️ Balanced User'

    archetype_stats_df['archetype_name'] = archetype_stats_df.apply(name_archetype, axis=1)
    return archetype_stats_df


# In your Power BI export section, add this:
# Group by the SOM cluster ID
archetype_summary = df_engineered.groupby('som_cluster').agg({
    'conversation_id': 'count',
    'engagement_time_sec': ['mean', 'median', 'std'],
    'confusion_score': 'mean',
    'trust_score': 'mean',
    'satisfaction_score': 'mean',
    'sentiment_compound': 'mean',
    'total_turns': 'mean',
    'user_message_length': 'mean'
}).round(3)

archetype_summary.columns = ['_'.join(col).strip() for col in archetype_summary.columns.values]
archetype_summary = archetype_summary.reset_index()

# ✨ APPLY MEANINGFUL NAMES HERE
archetype_summary = assign_behavioral_archetype_names(archetype_summary)

# Create mapping and update main dataframe
archetype_mapping = dict(zip(archetype_summary['som_cluster'],
                             archetype_summary['archetype_name']))
df_engineered['archetype_name'] = df_engineered['som_cluster'].map(archetype_mapping)

# Now export with proper names
archetype_summary.to_csv('powerbi_archetype_summary.csv', index=False)
df_engineered.to_csv('powerbi_interaction_data.csv', index=False)

In [55]:
print("\n📊 SECTION 12: POWER BI DASHBOARD DATA PREPARATION")
print("=" * 80)

# 1. Interaction-Level Export
df_powerbi_interactions = df_engineered[[
    'conversation_id', 'turn_number', 'user_id_hashed', 'timestamp',
    'country', 'user_intent', 'user_emotion', 'topic_name',
    'engagement_time_sec', 'confusion_score', 'trust_score', 'satisfaction_score',
    'sentiment_compound', 'sentiment_positive', 'sentiment_negative',
    'som_cluster', 'archetype_name', 'umap_1', 'umap_2',
    'total_turns', 'hour_of_day', 'is_weekend'
]].copy()

df_powerbi_interactions.to_csv('powerbi_interaction_data.csv', index=False)
print("\n✅ Exported: powerbi_interaction_data.csv")

# 2. Archetype Summary Export
archetype_summary = df_engineered.groupby('archetype_name').agg({
    'conversation_id': 'count',
    'engagement_time_sec': ['mean', 'median', 'std'],
    'confusion_score': 'mean',
    'trust_score': 'mean',
    'satisfaction_score': 'mean',
    'sentiment_compound': 'mean',
    'total_turns': 'mean',
    'user_message_length': 'mean'
}).round(3)

archetype_summary.columns = ['_'.join(col).strip() for col in archetype_summary.columns.values]
archetype_summary = archetype_summary.reset_index()
archetype_summary.to_csv('powerbi_archetype_summary.csv', index=False)
print("✅ Exported: powerbi_archetype_summary.csv")

# 3. Temporal Trends Export
df_engineered['date'] = df_engineered['timestamp'].dt.date
temporal_trends = df_engineered.groupby('date').agg({
    'conversation_id': 'count',
    'satisfaction_score': 'mean',
    'trust_score': 'mean',
    'confusion_score': 'mean',
    'sentiment_compound': 'mean',
    'engagement_time_sec': 'mean'
}).round(3).reset_index()

temporal_trends.to_csv('powerbi_temporal_trends.csv', index=False)
print("✅ Exported: powerbi_temporal_trends.csv")

# 4. Transition Network Export
transition_data = []
for i in range(len(top_transitions)):
    from_arch, to_arch, count, prob = top_transitions[i]
    from_name = archetype_names.get(from_arch, f"Archetype {from_arch}")
    to_name = archetype_names.get(to_arch, f"Archetype {to_arch}")
    transition_data.append({
        'from_archetype': from_name,
        'to_archetype': to_name,
        'transition_count': int(count),
        'transition_probability': round(prob, 4)
    })

df_transitions = pd.DataFrame(transition_data)
df_transitions.to_csv('powerbi_transition_network.csv', index=False)
print("✅ Exported: powerbi_transition_network.csv")

# 5. Topic-Archetype Matrix Export
topic_archetype = df_engineered.groupby(['archetype_name', 'topic_name']).size().reset_index(name='count')
topic_archetype_pivot = topic_archetype.pivot(index='archetype_name', columns='topic_name', values='count').fillna(0)
topic_archetype_pivot.to_csv('powerbi_topic_archetype_matrix.csv')
print("✅ Exported: powerbi_topic_archetype_matrix.csv")

print("=" * 80)


📊 SECTION 12: POWER BI DASHBOARD DATA PREPARATION

✅ Exported: powerbi_interaction_data.csv
✅ Exported: powerbi_archetype_summary.csv
✅ Exported: powerbi_temporal_trends.csv
✅ Exported: powerbi_transition_network.csv
✅ Exported: powerbi_topic_archetype_matrix.csv


VALIDATION & ETHICAL ASSESSMENT

In [56]:
print("\n✅ SECTION 13: VALIDATION & ETHICAL ASSESSMENT")
print("=" * 80)

print("""
\n🔍 Validation Metrics Summary:
================================
""")

# Clustering validation
print("📊 Clustering Quality:")
print(f"   - SOM Silhouette Score: {silhouette_som:.4f}")
print(f"   - KMeans Silhouette Score: {silhouette_kmeans:.4f}")
print(f"   - Number of Archetypes: {len(archetype_names)}")
print(f"   - Largest Archetype Coverage: {df_engineered['som_cluster'].value_counts().iloc[0] / len(df_engineered) * 100:.1f}%")

# Statistical model validation
print(f"\n📈 Satisfaction Model Performance:")
print(f"   - R-squared: {satisfaction_model.rsquared:.4f}")
print(f"   - Adjusted R-squared: {satisfaction_model.rsquared_adj:.4f}")
print(f"   - F-statistic: {satisfaction_model.fvalue:.2f}")
print(f"   - Prob (F-statistic): {satisfaction_model.f_pvalue:.2e}")

# Archetype stability
print(f"\n🔄 Archetype Transition Analysis:")
print(f"   - Users tracked: {len(archetype_sequences)}")
print(f"   - Total transitions recorded: {int(transition_counts.sum())}")
print(f"   - Most stable archetype: {transition_matrix.diagonal().argmax()}")

print("=" * 80)


✅ SECTION 13: VALIDATION & ETHICAL ASSESSMENT


🔍 Validation Metrics Summary:

📊 Clustering Quality:
   - SOM Silhouette Score: 0.0522
   - KMeans Silhouette Score: 0.0850
   - Number of Archetypes: 10
   - Largest Archetype Coverage: 3.7%

📈 Satisfaction Model Performance:
   - R-squared: 0.6191
   - Adjusted R-squared: 0.6191
   - F-statistic: 10200.10
   - Prob (F-statistic): 0.00e+00

🔄 Archetype Transition Analysis:
   - Users tracked: 4629
   - Total transitions recorded: 739
   - Most stable archetype: 9


SUMMARY STATISTICS & KEY FINDINGS

In [58]:
print("\n📑 SECTION 14: SUMMARY STATISTICS & KEY FINDINGS")
print("=" * 80)

print(f"""
\n🎯 PROJECT COMPLETION SUMMARY
===============================

DATASET OVERVIEW:
  Total Interactions: {len(df_engineered):,}
  Unique Conversations: {df_engineered['conversation_id'].nunique():,}
  Unique Users: {df_engineered['user_id_hashed'].nunique():,}
  Date Range: {df_engineered['timestamp'].min().date()} to {df_engineered['timestamp'].max().date()}
  Countries Represented: {df_engineered['country'].nunique()}
  User Intents Captured: {df_engineered['user_intent'].nunique()}

BEHAVIORAL GENOME:
  Behavioral Archetypes Discovered: {len(archetype_names)}
  Most Common Archetype: {df_engineered['archetype_name'].value_counts().index[0]}
  Archetype Coverage: {df_engineered['archetype_name'].value_counts().iloc[0] / len(df_engineered) * 100:.1f}%

ENGAGEMENT METRICS:
  Average Engagement Time: {df_engineered['engagement_time_sec'].mean():.1f} seconds
  Average Turns per Conversation: {df_engineered['total_turns'].mean():.1f}
  Median Satisfaction Score: {df_engineered['satisfaction_score'].median():.3f}
  Median Trust Score: {df_engineered['trust_score'].median():.3f}

SENTIMENT ANALYSIS:
  Average Sentiment: {df_engineered['sentiment_compound'].mean():.3f}
  Positive Interactions: {(df_engineered['sentiment_compound'] > 0.2).sum() / len(df_engineered) * 100:.1f}%
  Negative Interactions: {(df_engineered['sentiment_compound'] < -0.2).sum() / len(df_engineered) * 100:.1f}%
  Neutral Interactions: {((df_engineered['sentiment_compound'] >= -0.2) & (df_engineered['sentiment_compound'] <= 0.2)).sum() / len(df_engineered) * 100:.1f}%
""")

print("=" * 80)



📑 SECTION 14: SUMMARY STATISTICS & KEY FINDINGS


🎯 PROJECT COMPLETION SUMMARY

DATASET OVERVIEW:
  Total Interactions: 37,656
  Unique Conversations: 5,000
  Unique Users: 5,000
  Date Range: 2024-01-01 to 2024-12-30
  Countries Represented: 10
  User Intents Captured: 10

BEHAVIORAL GENOME:
  Behavioral Archetypes Discovered: 10
  Most Common Archetype: ✅ Efficient Collaborator
  Archetype Coverage: 21.4%

ENGAGEMENT METRICS:
  Average Engagement Time: 149.8 seconds
  Average Turns per Conversation: 8.7
  Median Satisfaction Score: 0.500
  Median Trust Score: 0.500

SENTIMENT ANALYSIS:
  Average Sentiment: -0.003
  Positive Interactions: 34.0%
  Negative Interactions: 34.4%
  Neutral Interactions: 31.6%

